In [ ]:
!pip install nucleus-cdk==0.5.0rc2 | tail -n2

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = pd.read_csv("../raw-data/20260305_round_0.csv")
# df.loc[0, "experiment_name"] = "id"

In [3]:
column_types = pd.DataFrame(data=np.reshape(["feature",]*len(df.columns), (1, len(df.columns))), columns=df.columns)

In [4]:
column_types.experiment_id.iloc[0] = 'id'
column_types.success.iloc[0] = 'classifier'
column_types.loc[0,'sigmoid_steady_state (ng/uL)':'sigmoid_time_offset (h)'] = 'regressor'

In [5]:
column_types

,experiment_id,Plate,Well,Read,Experiment,Name,Type,PMix ID,Ribosome ID,SMS ID,...,Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,...,feature,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier


In [6]:
df = pd.concat([column_types, df],axis=0,ignore_index=True)

In [7]:
df

,experiment_id,Plate,Well,Read,Experiment,Name,Type,PMix ID,Ribosome ID,SMS ID,...,Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,...,feature,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier
1,PMix-AR-898_B2,1,B2,GFP-M-Gext,PMix-AR-898,AR-898 (16.25 mg/mL),Sample,AR-898,AR-898,AR-895,...,10.0,8.0,20.0,0.0,0.0,100.0,35.506085,3.128813,1.352958,True
2,PMix-AR-898_B4,1,B4,GFP-M-Gext,PMix-AR-898,AR-898 (15 mg/mL),Sample,AR-898,AR-898,AR-895,...,10.0,8.0,20.0,0.0,0.0,100.0,35.474534,3.431828,1.281995,True
3,PMix-AR-898_B10,1,B10,GFP-M-Gext,PMix-AR-898,AR-898 (15 mg/mL) + BE Rib (12.14 uM),Sample,AR-898,AR-898.1,AR-895,...,10.0,8.0,20.0,0.0,0.0,100.0,44.145432,2.785779,1.414129,True
4,PMix-AR-898_B12,1,B12,GFP-M-Gext,PMix-AR-898,AR-898 (15 mg/mL) + BE Rib (10 uM),Sample,AR-898,AR-898.1,AR-895,...,10.0,8.0,20.0,0.0,0.0,100.0,39.114225,2.541942,1.495824,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,Mg_K_screen-20251105_090558_O4,8,O4,GFP-F-G35,Mg_K_screen-20251105_090558,Mg 8; K 34,Sample,NaN,NaN,NaN,...,5.0,9.0,20.0,0.0,0.0,120.0,26.736498,2.162077,1.4333,True
189,Mg_K_screen-20251105_090558_P1,8,P1,GFP-F-G35,Mg_K_screen-20251105_090558,Mg 4; K 17,Sample,NaN,NaN,NaN,...,5.0,7.0,20.0,0.0,0.0,100.0,35.942818,2.088428,1.439968,True
190,Mg_K_screen-20251105_090558_P2,8,P2,GFP-F-G35,Mg_K_screen-20251105_090558,Mg 4; K 34,Sample,NaN,NaN,NaN,...,5.0,7.0,20.0,0.0,0.0,120.0,46.248301,2.120363,1.438154,True
191,Mg_K_screen-20251105_090558_P3,8,P3,GFP-F-G35,Mg_K_screen-20251105_090558,Mg 4; K 51,Sample,NaN,NaN,NaN,...,5.0,7.0,20.0,0.0,0.0,140.0,44.246019,2.091836,1.455216,True


First, pick only the monochromator reads on Cytation 3, and then drop read information:

In [8]:
df = df.drop(df.query("Reader == 'Cytation3' and Read == 'GFP-F-G35'").index)

In [9]:
df.loc[1:,:].groupby(['Reader','Read']).Read.count()

Reader     Read      
Cytation3  GFP-M-G100    48
Cytation5  GFP-M-Gext    36
Synergy 2  GFP-F-G35     60
Name: Read, dtype: int64

Drop columns that aren't going to be used in the fitting:

In [143]:
df.columns

Index(['experiment_id', 'Plate', 'Well', 'Read', 'Experiment', 'Name', 'Type',
       'PMix ID', 'Ribosome ID', 'SMS ID', 'tRNA ID', 'DNA ID', 'SMS Vol (uL)',
       'Water vol (uL)', 'Reader', 'HasCP', 'Gain', 'Read Type', 'Read_new',
       '[HEPES] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[tRNA] (ug/uL)', '[TCEP] (mM)', '[Folinic acid] (mM)',
       '[Spermidine] (mM)', '[Amino acid mix] (mM)', '[DNA] (ng/uL)',
       'DNA MW (MDa)', '[DNA] (nM)', 'Product', '[PEG4K 40%] (%)',
       '[RNAse Inhib stock] (U/mL)', '[RNAse Inhib] (U/mL)', 'IsNEB', 'Date',
       '[PMix stock] (mg/mL)', '[Ribosome stock] (uM)', '[tRNA stock] (ug/uL)',
       '[DNA stock] (ng/uL)', 'PMix Vol (uL)', '[PMix] (mg/mL)',
       'Ribosome Vol (uL)', '[Ribosome] (uM)', 'tRNA Vol (uL)', 'DNA Vol (uL)',
       'RNase Inhib Vol (uL)', 'Rxn Volume (uL)', '[Magnesium acetate] (mM)',
       '[Creatine phosphate] (mM)', '[PPK] (uM)', '[PolyP] (mM)',
       '[Potassium glutamate] (mM)', '

In [10]:
df.drop(['Plate', 'Read', 'Well', 'Type','Experiment', 'Name', 'PMix ID', 'Ribosome ID', 'SMS ID', 'tRNA ID', 'DNA ID', 'SMS Vol (uL)', 'Water vol (uL)', 'Reader', 'HasCP', 'Gain', 'Read Type', 'Read_new', '[DNA] (ng/uL)', 'DNA MW (MDa)', 
         '[RNAse Inhib stock] (U/mL)', 'Date', '[PMix stock] (mg/mL)', '[Ribosome stock] (uM)', '[tRNA stock] (ug/uL)', '[DNA stock] (ng/uL)', 'PMix Vol (uL)', 'Ribosome Vol (uL)','tRNA Vol (uL)', 'DNA Vol (uL)', 'RNase Inhib Vol (uL)',
       ],axis=1, inplace=True) #  'Rxn Volume (uL)'

In [11]:
## Convert categorical "Product" to one hot (which is just plamGFP or not (= deGFP))
product_bool = (df["Product"].iloc[1:] == "plamGFP").astype(int)
product_bool.loc[0] = "feature"
df["isplamGFP"] = product_bool
df = df.drop("Product", axis=1)

In [ ]:
# real_one_hot = pd.get_dummies(df["Experiment"].iloc[1:]).astype(int)
# real_one_hot.loc[0] = "feature"
# # df = pd.concat([df, real_one_hot], axis=1).drop("Read", axis=1)
# real_one_hot

In [13]:
df.to_csv('20260305-round-0-allcols.csv',index=False)

In [45]:
# Drop columns that have only constant values
unique_cols = df.columns[df[1:].nunique() == 1]
df = df.drop(unique_cols, axis=1)

In [30]:
unique_cols

Index(['[HEPES] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[tRNA] (ug/uL)', '[TCEP] (mM)', '[Folinic acid] (mM)',
       '[Spermidine] (mM)', '[Amino acid mix] (mM)'],
      dtype='object')

In [46]:
df

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),IsNEB,[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier,feature
1,PMix-AR-898_B2,3.376553,0.0,2000.0,0.0,1.95,1.8,10.0,8.0,20.0,0.0,0.0,100.0,35.506085,3.128813,1.352958,True,1
2,PMix-AR-898_B4,3.376553,0.0,2000.0,0.0,1.8,1.8,10.0,8.0,20.0,0.0,0.0,100.0,35.474534,3.431828,1.281995,True,1
3,PMix-AR-898_B10,3.376553,0.0,2000.0,0.0,1.8,2.1852,10.0,8.0,20.0,0.0,0.0,100.0,44.145432,2.785779,1.414129,True,1
4,PMix-AR-898_B12,3.376553,0.0,2000.0,0.0,1.8,1.8,10.0,8.0,20.0,0.0,0.0,100.0,39.114225,2.541942,1.495824,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,Mg_K_screen-20251105_090558_O4,5.0,0.0,2000.0,1.0,1.8,1.8,5.0,9.0,20.0,0.0,0.0,120.0,26.736498,2.162077,1.4333,True,1
189,Mg_K_screen-20251105_090558_P1,5.0,0.0,2000.0,1.0,1.8,1.8,5.0,7.0,20.0,0.0,0.0,100.0,35.942818,2.088428,1.439968,True,1
190,Mg_K_screen-20251105_090558_P2,5.0,0.0,2000.0,1.0,1.8,1.8,5.0,7.0,20.0,0.0,0.0,120.0,46.248301,2.120363,1.438154,True,1
191,Mg_K_screen-20251105_090558_P3,5.0,0.0,2000.0,1.0,1.8,1.8,5.0,7.0,20.0,0.0,0.0,140.0,44.246019,2.091836,1.455216,True,1


In [47]:
df.to_csv("20260305_round_0.csv", index=False)

For setting bounds, keep track of maximum and minimum values:

In [48]:
numerical_vals_df = df.loc[1:, '[DNA] (nM)':].apply(pd.to_numeric)

In [49]:
numerical_vals_df

,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),IsNEB,[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
1,3.376553,0.0,2000.0,0.0,1.95,1.8000,10.0,8.0,20.0,0.0,0.0,100.0,35.506085,3.128813,1.352958,True,1
2,3.376553,0.0,2000.0,0.0,1.80,1.8000,10.0,8.0,20.0,0.0,0.0,100.0,35.474534,3.431828,1.281995,True,1
3,3.376553,0.0,2000.0,0.0,1.80,2.1852,10.0,8.0,20.0,0.0,0.0,100.0,44.145432,2.785779,1.414129,True,1
4,3.376553,0.0,2000.0,0.0,1.80,1.8000,10.0,8.0,20.0,0.0,0.0,100.0,39.114225,2.541942,1.495824,True,1
5,3.376553,0.0,2000.0,0.0,1.95,1.8000,10.0,8.0,20.0,0.0,0.0,100.0,38.056368,3.493323,1.265536,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,5.000000,0.0,2000.0,1.0,1.80,1.8000,5.0,9.0,20.0,0.0,0.0,120.0,26.736498,2.162077,1.433300,True,1
189,5.000000,0.0,2000.0,1.0,1.80,1.8000,5.0,7.0,20.0,0.0,0.0,100.0,35.942818,2.088428,1.439968,True,1
190,5.000000,0.0,2000.0,1.0,1.80,1.8000,5.0,7.0,20.0,0.0,0.0,120.0,46.248301,2.120363,1.438154,True,1
191,5.000000,0.0,2000.0,1.0,1.80,1.8000,5.0,7.0,20.0,0.0,0.0,140.0,44.246019,2.091836,1.455216,True,1


In [50]:
minmax_df = pd.DataFrame({'min': numerical_vals_df.min(axis=0), 'max': numerical_vals_df.max(axis=0)}).T # 

In [51]:
minmax_df

,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),IsNEB,[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
min,2.996187,0.0,0.0,0.0,1.8,1.8,5.0,6.0,0.0,0.0,0.0,100.0,0.024858,1.367521,0.743901,False,0
max,5.0,2.0,2000.0,1.0,1.95,3.24,10.0,20.0,20.0,2.0,30.0,140.0,114.117675,5.84127,3.760556,True,1


In [52]:
minmax_df['[Ribosome] (uM)']

min     1.8
max    3.24
Name: [Ribosome] (uM), dtype: object

In [53]:
minmax_df.to_csv("dataset-20260305-min-max.csv")